In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

# Đọc dữ liệu đúng tên file trên Colab (có đuôi .csv)
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv').iloc[:, 0]

X_val = pd.read_csv('X_validation.csv')
y_val = pd.read_csv('y_validation.csv').iloc[:, 0]

print(f"Đọc dữ liệu thành công! X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# Tuning Random Forest
rf = RandomForestRegressor(random_state=42)
rf_params = {'n_estimators': [50, 100], 'max_depth': [10, 20]}
rf_search = RandomizedSearchCV(estimator=rf, param_distributions=rf_params,
                               n_iter=2, cv=3, scoring='neg_root_mean_squared_error',
                               random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

# Tuning Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb_params = {'n_estimators': [50, 100], 'learning_rate': [0.1], 'max_depth': [3, 5]}
gb_search = RandomizedSearchCV(estimator=gb, param_distributions=gb_params,
                               n_iter=2, cv=3, scoring='neg_root_mean_squared_error',
                               random_state=42, n_jobs=-1)
gb_search.fit(X_train, y_train)
best_gb = gb_search.best_estimator_

# Chọn mô hình tốt nhất
models = {'Random Forest': best_rf, 'Gradient Boosting': best_gb}
best_score = float('inf')
final_best_model = None

for name, model in models.items():
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    print(f"\n{name} -> RMSE: {rmse:.4f}")
    if rmse < best_score:
        best_score = rmse
        final_best_model = model

# Lưu mô hình
joblib.dump(final_best_model, 'best_model.pkl')
print("\n==> Đã tạo và lưu file best_model.pkl thành công!")

Đọc dữ liệu thành công! X_train shape: (374511, 37), X_val shape: (106366, 37)

Random Forest -> RMSE: 89.1274

Gradient Boosting -> RMSE: 89.1295

==> Đã tạo và lưu file best_model.pkl thành công!
